## Reading the CSV

In [40]:
import pandas as pd

df = pd.read_csv('cars.csv')
df.head()

,url,price,brand,series,model,year,mileage,transmission,fuel,body,...,engine,hp,drive,condition,heavy_damage,paint_changed,fuel_consumption,fuel_tank,trade_in,seller_type
0,https://www.arabam.com/ilan/galeriden-satilik-...,1.049.000 TL,Ford,Focus,1.5 TDCi Trend X,2018.0,180.000 km,Otomatik,Dizel,Sedan,...,1401 - 1600 cm3,101 - 125 HP,Önden Çekiş,İkinci El,NaN,"1 değişen, 2 boyalı",NaN,NaN,Takasa Uygun,Galeriden
1,https://www.arabam.com/ilan/galeriden-satilik-...,555.750 TL,Ford,Focus,1.6 TDCi Collection,2010.0,357.000 km,Düz,Dizel,Hatchback/5,...,1560 cc,90 hp,Önden Çekiş,İkinci El,Hayır,5 boyalı,"4,7 lt",55 lt,Takasa Uygun,Galeriden
2,https://www.arabam.com/ilan/galeriden-satilik-...,595.000 TL,Citroen,C-Elysée,1.6 HDi Attraction,2014.0,225.000 km,Düz,Dizel,Sedan,...,1560 cc,93 hp,Önden Çekiş,İkinci El,NaN,2 boyalı,"4,3 lt",48 lt,Takasa Uygun,Galeriden
3,https://www.arabam.com/ilan/galeriden-satilik-...,2.059.000 TL,Mercedes - Benz,C,C 180 BlueEFFICIENCY AMG,2014.0,75.000 km,Otomatik,Benzin,Coupe,...,1401 - 1600 cm3,151 - 175 HP,Arkadan İtiş,İkinci El,Belirtilmemiş,Tamamı orjinal,NaN,NaN,Takasa Uygun,Galeriden
4,https://www.arabam.com/ilan/galeriden-satilik-...,1.150.000 TL,Volvo,S60,1.6 D Advance,2014.0,193.000 km,Otomatik,Dizel,Sedan,...,1401 - 1600 cm3,101 - 125 HP,Önden Çekiş,İkinci El,NaN,1 değişen,NaN,NaN,Takasa Uygun,Galeriden


## Cleaning the Data

### We want to clean the price, mileage, and year data to convert them to floats and drop the unrealistic data.

In [41]:
df = df.dropna(subset=["price", "brand", "series", "year", "mileage", "hp"])

In [42]:
df = df.drop_duplicates(subset=["url"])

In [43]:
df["price"] = (
    df["price"]
    .str.replace(" TL", "", regex=False)
    .str.replace(".", "", regex=False)
    .astype(float)
)

df = df[(df["price"] < 20000000) & (df["price"] > 100000)].copy()

df["price"].describe()

count    4.138000e+04
mean     1.106251e+06
std      1.101223e+06
min      1.025000e+05
25%      5.250000e+05
50%      8.550000e+05
75%      1.319962e+06
max      1.998500e+07
Name: price, dtype: float64

In [44]:
df["mileage"] = df["mileage"].str.replace(" km", "", regex=False).str.replace(".", "", regex=False).astype(float)

df = df[
    (df["year"] >= 1980) &
    (df["year"] <= 2026) &
    (df["mileage"] > 0) &
    (df["mileage"] < 1_000_000)
]

df["mileage"].describe()

count     41340.000000
mean     190334.342937
std      103165.567538
min        5080.000000
25%      114894.250000
50%      185000.000000
75%      256000.000000
max      999999.000000
Name: mileage, dtype: float64

In [45]:
df['heavy_damage'] = df['heavy_damage'].fillna('Belirtilmemiş')
df["heavy_damage"].describe()

count             41340
unique                3
top       Belirtilmemiş
freq              26379
Name: heavy_damage, dtype: object

In [46]:
df["hp"] = (
    df["hp"]
    .str.lower()
    .str.replace(" hp", "", regex=False)
    .str.strip()
    .str.split(" - ", expand=True)
    .apply(pd.to_numeric, errors="coerce")
    .astype(float)
    .mean(axis=1)
)
df = df.dropna(subset=["hp"])

In [47]:
df = df.dropna(subset=["paint_changed"])
df = df[df["paint_changed"] != "Belirtilmemiş"]

df["degisen"] = df["paint_changed"].str.extract(r"(\d+)\s+değişen").fillna(0).astype(int)
df["lokal_boyali"] = df["paint_changed"].str.extract(r"(\d+)\s+lokal\s+boyalı").fillna(0).astype(int)

boyali_only = df["paint_changed"].str.replace(r"\d+\s+lokal\s+boyalı", "", regex=True)
df["boyali"] = boyali_only.str.extract(r"(\d+)\s+boyalı").fillna(0).astype(int)

df.loc[df["paint_changed"] == "Tamamı boyalı", "boyali"] = 12
df.loc[df["paint_changed"] == "Tamamı lokal boyalı", "lokal_boyali"] = 12

In [48]:
df = df.reset_index(drop=True)
df.head()

,url,price,brand,series,model,year,mileage,transmission,fuel,body,...,condition,heavy_damage,paint_changed,fuel_consumption,fuel_tank,trade_in,seller_type,degisen,lokal_boyali,boyali
0,https://www.arabam.com/ilan/galeriden-satilik-...,1049000.0,Ford,Focus,1.5 TDCi Trend X,2018.0,180000.0,Otomatik,Dizel,Sedan,...,İkinci El,Belirtilmemiş,"1 değişen, 2 boyalı",NaN,NaN,Takasa Uygun,Galeriden,1,0,2
1,https://www.arabam.com/ilan/galeriden-satilik-...,555750.0,Ford,Focus,1.6 TDCi Collection,2010.0,357000.0,Düz,Dizel,Hatchback/5,...,İkinci El,Hayır,5 boyalı,"4,7 lt",55 lt,Takasa Uygun,Galeriden,0,0,5
2,https://www.arabam.com/ilan/galeriden-satilik-...,595000.0,Citroen,C-Elysée,1.6 HDi Attraction,2014.0,225000.0,Düz,Dizel,Sedan,...,İkinci El,Belirtilmemiş,2 boyalı,"4,3 lt",48 lt,Takasa Uygun,Galeriden,0,0,2
3,https://www.arabam.com/ilan/galeriden-satilik-...,2059000.0,Mercedes - Benz,C,C 180 BlueEFFICIENCY AMG,2014.0,75000.0,Otomatik,Benzin,Coupe,...,İkinci El,Belirtilmemiş,Tamamı orjinal,NaN,NaN,Takasa Uygun,Galeriden,0,0,0
4,https://www.arabam.com/ilan/galeriden-satilik-...,1150000.0,Volvo,S60,1.6 D Advance,2014.0,193000.0,Otomatik,Dizel,Sedan,...,İkinci El,Belirtilmemiş,1 değişen,NaN,NaN,Takasa Uygun,Galeriden,1,0,0


## Concatenate Brand & Series

### I decided to concatenate these two since each brand's series are special to that brand.

In [49]:
df["brand_series"] = df["brand"] + "_" + df["series"]
df["brand_series"].describe()

count              37872
unique               281
top       Toyota_Corolla
freq                1726
Name: brand_series, dtype: object

## Encoding Brand & Series

### We want to encode these to labels for our model to understand since working with strings does not make sense.

In [50]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["brand_series_encoded"] = le.fit_transform(df["brand_series"])
df["heavy_damage_encoded"] = le.fit_transform(df["heavy_damage"])
print(df["brand_series_encoded"].head())
print(df["heavy_damage_encoded"].head())

0     74
1     74
2     27
3    138
4    270
Name: brand_series_encoded, dtype: int64
0    0
1    2
2    0
3    0
4    0
Name: heavy_damage_encoded, dtype: int64


In [51]:
df = df.replace({"body": {"-": pd.NA}, "drive": {"-": pd.NA}})
df = df.dropna(subset=["body", "fuel", "transmission", "drive"])

for col in ["body", "fuel", "transmission", "drive"]:
    df[col + "_encoded"] = LabelEncoder().fit_transform(df[col])

## Creating the Train & Test Splits

In [52]:
X = df[['brand_series_encoded', 'year', 'mileage', 'heavy_damage_encoded', 'hp', 'degisen', 'boyali', 'lokal_boyali', 'body_encoded', 'fuel_encoded', 'transmission_encoded', 'drive_encoded']]
y = df["price"]

print(X.head())
print(y.head())

   brand_series_encoded    year   mileage  heavy_damage_encoded     hp  \
0                    74  2018.0  180000.0                     0  113.0   
1                    74  2010.0  357000.0                     2   90.0   
2                    27  2014.0  225000.0                     0   93.0   
3                   138  2014.0   75000.0                     0  163.0   
4                   270  2014.0  193000.0                     0  113.0   

   degisen  boyali  lokal_boyali  body_encoded  fuel_encoded  \
0        1       2             0             8             1   
1        0       5             0             3             1   
2        0       2             0             8             1   
3        0       0             0             1             0   
4        1       0             0             8             1   

   transmission_encoded  drive_encoded  
0                     1              3  
1                     0              3  
2                     0              3  
3     

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

## Training the baseline decision tree

In [54]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print("Decision Tree MAE:", mean_absolute_error(y_test, y_pred))

Decision Tree MAE: 153066.45148462354


## Training the baseline XGBoost

In [55]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print("XGBoost MAE:", mean_absolute_error(y_test, y_pred_xgb))

XGBoost MAE: 101000.17335382423


## Training the baseline LightGBM

In [56]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    objective="regression",
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

categorical_features = ["brand_series_encoded", "heavy_damage_encoded", "body_encoded", "fuel_encoded", "transmission_encoded", "drive_encoded"]
lgbm.fit(X_train, y_train, categorical_feature=categorical_features)

y_pred_lgbm = lgbm.predict(X_test)
print("LightGBM MAE:", mean_absolute_error(y_test, y_pred_lgbm))


LightGBM MAE: 86695.6700601116


## Randomized Search for finding better hyperparameters for LightGBM

In [57]:
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
from scipy.stats import randint, uniform

baseline_lgbm_params = {
    "n_estimators": 700,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
}

random_param_dist = {
    "n_estimators": randint(300, 1501),
    "learning_rate": uniform(0.01, 0.1),
    "num_leaves": randint(15, 81),
    "max_depth": [-1, 6, 8, 10, 12],
    "min_child_samples": randint(10, 61),
}

random_candidates = list(ParameterSampler(random_param_dist, n_iter=50, random_state=42))
param_candidates = [
    {key: [value] for key, value in baseline_lgbm_params.items()},
    *[
        {key: [value] for key, value in params.items()}
        for params in random_candidates
        if params != baseline_lgbm_params
    ],
]

search = RandomizedSearchCV(
    LGBMRegressor(objective="regression", random_state=42, verbose=-1),
    param_distributions=param_candidates,
    n_iter=len(param_candidates),
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train, categorical_feature=categorical_features)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMRegressor...2, verbose=-1)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'learning_rate': [0.05], 'max_depth': [-1], 'min_child_samples': [20], 'n_estimators': [700], ...}, {'learning_rate': [np.float64(0....4011884736254)], 'max_depth': [12], 'min_child_samples': [24], 'n_estimators': [1430], ...}, ...]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",51
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits

## Testing the tuned LightGBM on the test set

In [58]:
cv_results = pd.DataFrame(search.cv_results_)
baseline_mask = pd.Series(True, index=cv_results.index)
for param_name, baseline_value in baseline_lgbm_params.items():
    baseline_mask &= cv_results[f"param_{param_name}"] == baseline_value

baseline_cv_mae = -cv_results.loc[baseline_mask, "mean_test_score"].iloc[0]
y_pred_tuned = search.best_estimator_.predict(X_test)

print("Baseline candidate CV MAE:", baseline_cv_mae)
print("Best CV MAE:", -search.best_score_)
print("Baseline LightGBM Test MAE:", mean_absolute_error(y_test, y_pred_lgbm))
print("Tuned LightGBM Test MAE:", mean_absolute_error(y_test, y_pred_tuned))
print("Best params:", search.best_params_)

Baseline candidate CV MAE: 93741.24826290166
Best CV MAE: 92427.88376718557
Baseline LightGBM Test MAE: 86695.6700601116
Tuned LightGBM Test MAE: 86019.72120675107
Best params: {'num_leaves': 33, 'n_estimators': 1422, 'min_child_samples': 18, 'max_depth': 8, 'learning_rate': np.float64(0.06582934536070977)}


## Test-set bargain finder

Rank only listings from the held-out test set, so every displayed bargain is a listing the model was not trained on. Also keep reliability flags visible so rare brand-series groups and older cars can be reviewed more cautiously.

In [62]:
MIN_BRAND_SERIES_TRAIN_COUNT = 20
OLD_CAR_YEAR_CUTOFF = 2010

train_brand_series_counts = df.loc[X_train.index, "brand_series"].value_counts()

test_bargains = df.loc[X_test.index].copy()
test_bargains["predicted_price"] = lgbm.predict(X_test)
test_bargains["discount"] = test_bargains["predicted_price"] - test_bargains["price"]
test_bargains["discount_pct"] = test_bargains["discount"] / test_bargains["predicted_price"]
test_bargains["brand_series_train_count"] = (
    test_bargains["brand_series"].map(train_brand_series_counts).fillna(0).astype(int)
)
test_bargains["rare_brand_series"] = test_bargains["brand_series_train_count"] < MIN_BRAND_SERIES_TRAIN_COUNT
test_bargains["old_car"] = test_bargains["year"] <= OLD_CAR_YEAR_CUTOFF

def reliability_flags(row):
    flags = []
    if row["rare_brand_series"]:
        flags.append("rare_brand_series")
    if row["old_car"]:
        flags.append("old_car")
    return ", ".join(flags) or ""

test_bargains["reliability_flags"] = test_bargains.apply(reliability_flags, axis=1)

bargain_columns = [
    "url", "brand_series", "model", "year", "mileage", "price", "predicted_price",
    "discount", "discount_pct", "brand_series_train_count", "reliability_flags",
    "heavy_damage", "paint_changed",
]

bargains = test_bargains[bargain_columns].sort_values("discount_pct", ascending=False)
reliable_bargains = bargains[bargains["brand_series_train_count"] >= MIN_BRAND_SERIES_TRAIN_COUNT]

print(f"Test-set bargain candidates: {len(bargains):,}")
print(f"Reliable candidates with at least {MIN_BRAND_SERIES_TRAIN_COUNT} training comparables: {len(reliable_bargains):,}")
print(f"Rare brand-series candidates filtered out: {len(bargains) - len(reliable_bargains):,}")

reliable_bargains.head(20)

Test-set bargain candidates: 7,544
Reliable candidates with at least 20 training comparables: 7,334
Rare brand-series candidates filtered out: 210


,url,brand_series,model,year,mileage,price,predicted_price,discount,discount_pct,brand_series_train_count,reliability_flags,heavy_damage,paint_changed
23774,https://www.arabam.com/ilan/sahibinden-satilik...,Citroen_C5,1.6 e-HDi Confort,2011.0,500000.0,154000.0,4.632821e+05,3.092821e+05,0.667589,105,,Hayır,"5 boyalı, 1 lokal boyalı"
11998,https://www.arabam.com/ilan/sahibinden-satilik...,Ford_Escort,1.6 CL,1997.0,46571.0,119000.0,3.158221e+05,1.968221e+05,0.623206,38,old_car,Hayır,Tamamı orjinal
32403,https://www.arabam.com/ilan/sahibinden-satilik...,Citroen_Xsara,1.8 SX,1998.0,145000.0,145000.0,3.839406e+05,2.389406e+05,0.622337,24,old_car,Hayır,1 değişen
33044,https://www.arabam.com/ilan/sahibinden-satilik...,Nissan_Primera,2.0 GT,1999.0,336000.0,175000.0,3.904756e+05,2.154756e+05,0.551829,118,old_car,Hayır,Tamamı orjinal
26487,https://www.arabam.com/ilan/galeriden-satilik-...,Citroen_Xsara,1.6 SX,1998.0,339000.0,175000.0,3.882341e+05,2.132341e+05,0.549241,24,old_car,Belirtilmemiş,Tamamı orjinal
20267,https://www.arabam.com/ilan/sahibinden-satilik...,Honda_Accord,2.0 EX,1992.0,340000.0,175000.0,3.669311e+05,1.919311e+05,0.523071,40,old_car,Belirtilmemiş,Tamamı orjinal
17991,https://www.arabam.com/ilan/sahibinden-satilik...,Peugeot_306,1.8 XR,1997.0,370000.0,105000.0,2.058975e+05,1.008975e+05,0.490038,29,old_car,Belirtilmemiş,12 boyalı
92,https://www.arabam.com/ilan/galeriden-satilik-...,Ford_Fiesta,1.4 TDCi Sport,2006.0,228000.0,225000.0,4.310728e+05,2.060728e+05,0.478046,457,old_car,Belirtilmemiş,Tamamı orjinal
5157,https://www.arabam.com/ilan/galeriden-satilik-...,Mercedes - Benz_E,200,1994.0,450000.0,315000.0,5.962298e+05,2.812298e+05,0.471680,539,old_car,Belirtilmemiş,1 lokal boyalı
2365,https://www.arabam.com/ilan/galeriden-satilik-...,BMW_3 Serisi,316i Compact,1998.0,266000.0,260500.0,4.897413e+05,2.292413e+05,0.468087,721,old_car,Belirtilmemiş,11 boyalı
